# Padded Splines
The data samples are represented with circles. The first sample, as well as its replicates when the padding is globally periodic, is indicated by a red stem line. The portion of curve in thick green is tied to the observed data, with a number of samples that can be specified. The portion of curve in thick blue is the complement to a full period.

In [1]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_degree = 9 # Maximal spline degree
max_samples = 12 # Maximal support

# Initial random periodic cubic spline
s0 = sk.PeriodicSpline1D.from_spline_coeff(np.random.standard_normal(6), degree = 3)

# Plot
def update_plot (
    degree = 3,
    samples = 6,
    basis = 0
):
    global s0

    # Update of the spline
    f = s0.get_samples(0, support_length = s0.period)
    if s0.period < samples:
        f = np.append(f, np.random.standard_normal(samples - len(f)))
    else:
        f = f[ : samples]
    s0 = sk.PeriodicSpline1D.from_samples(f, degree = degree)

    # Decomposition according to the basis
    subplot = plt.subplots()
    if 0 == basis: # BASIC
        weights = np.diag(s0.spline_coeff)
        bases = [sk.PeriodicSpline1D.from_spline_coeff(w, degree = degree) for w in weights]
        image = {b.image() for b in bases}
        image.add(s0.image())
        plotrange = sk.interval.Interval.enclosure(image)
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.55 * plotrange.diameter,
            plotrange.midpoint + 0.55 * plotrange.diameter
        ))
        s0.plot(
            subplot,
            plotrange = plotrange,
            plotpoints = 200 + 1,
            curve_fmt = "-C0",
#            curve_markerfmt = "",
            curvestem_linefmt = "None",
            knot_marker = "",
#            periodbound_markerfmt = "",
            periodboundstem_linefmt = "None"
        )
        for k in range(s0.period):
            sk.PeriodicSpline1D.from_spline_coeff(weights[k], degree = degree).plot(
                subplot,
#                plotrange = plotrange,
                plotpoints = 200 + 1,
                curve_fmt = "-C" + str(1 + k % 9),
                curve_lw = 0.25,
                curve_markerfmt = "",
                curvestem_linefmt = "None",
                knot_marker = "",
                periodbound_markerfmt = "",
                periodboundstem_linefmt = "None"
            )
    elif 1 == basis: # CARDINAL
        pass
    elif 2 == basis: # DUAL
        pass
    elif 3 == basis: # ORTHONORMAL
        pass
    plt.show()

widgets.interactive(
    update_plot,
    degree = (0, max_degree),
    samples = (1, max_samples),
    basis = widgets.RadioButtons(
        options = [
            ("BASIC", 0),
            ("CARDINAL", 1),
            ("DUAL", 2),
            ("ORTHONORMAL", 3)
        ],
        value = 0,
        description = "Basis:",
        disabled = False
    )
)


interactive(children=(IntSlider(value=3, description='degree', max=9), IntSlider(value=6, description='samples…